# Airbnb Berlin: факторы стоимости ночи

**Пет-проект / разведочный анализ данных**  
Данные: Inside Airbnb, объявления Берлина (2025).

Проект превращает курсовое исследование в воспроизводимый разведочный анализ данных (EDA): очистка, визуализации и проверка факторов, связанных со стоимостью ночи.

## 1. Настройка и загрузка данных

Ноутбук использует точный срез [Inside Airbnb — Berlin](https://insideairbnb.com/get-the-data/) от **23 сентября 2025 года**. Если файла нет рядом с ноутбуком, код скачает его автоматически.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

DATA_PATH = Path('listings.csv.gz')
DATA_URL = ('https://data.insideairbnb.com/germany/be/berlin/'
            '2025-09-23/data/listings.csv.gz')
if not DATA_PATH.exists():
    from urllib.request import urlretrieve
    print('Скачивается срез данных по Берлину от 23.09.2025…')
    urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)
print(f'Исходных строк: {df.shape[0]:,}; столбцов: {df.shape[1]:,}')
df.head()

## 2. Предобработка

Цена приводится к числовому формату; экстремальные значения выше 99-го перцентиля исключаются, чтобы не искажать сравнения.

In [ ]:
df = df.copy()
df['price'] = df['price'].replace(r'[\$,]', '', regex=True).astype(float)
p99 = df['price'].quantile(0.99)
df_clean = df.loc[df['price'].le(p99)].copy()

cols = [
    'price', 'room_type', 'neighbourhood_group_cleansed', 'accommodates',
    'bedrooms', 'bathrooms', 'beds', 'minimum_nights',
    'number_of_reviews', 'review_scores_rating', 'host_is_superhost',
    'availability_365'
]
df_clean = df_clean[cols]
print(f'99-й перцентиль: €{p99:,.0f}')
print(f'Строк после обработки выбросов: {len(df_clean):,}')
df_clean['price'].describe()

## 3. Распределение цены и тип жилья

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_clean['price'], bins=45, color='#3856A6', edgecolor='white')
axes[0].set(title='Распределение стоимости ночи', xlabel='Цена, €', ylabel='Количество объявлений')

room_order = ['Entire home/apt', 'Private room', 'Shared room', 'Hotel room']
sns.boxplot(data=df_clean, x='room_type', y='price', order=room_order, ax=axes[1], showfliers=False)
axes[1].set(title='Стоимость по типам жилья', xlabel='', ylabel='Цена, €')
axes[1].tick_params(axis='x', rotation=18)
fig.tight_layout()
fig.savefig(FIG_DIR / '01_price_and_room_type.png', dpi=180, bbox_inches='tight')
plt.show()

df_clean.groupby('room_type')['price'].agg(['count', 'mean', 'median']).sort_values('mean', ascending=False).round(1)

## 4. Локация и вместимость

In [ ]:
by_neigh = (df_clean.groupby('neighbourhood_group_cleansed')['price']
              .agg(['count', 'mean', 'median'])
              .sort_values('mean', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
by_neigh['mean'].sort_values().plot.barh(ax=axes[0], color='#2F7D6D')
axes[0].set(title='Средняя стоимость ночи по районам', xlabel='Цена, €', ylabel='')

sns.scatterplot(data=df_clean, x='accommodates', y='price', alpha=.22, s=18, ax=axes[1], color='#3856A6')
axes[1].set(title='С ростом вместимости растёт цена', xlabel='Количество гостей', ylabel='Цена, €')
fig.tight_layout()
fig.savefig(FIG_DIR / '02_location_and_capacity.png', dpi=180, bbox_inches='tight')
plt.show()

by_neigh.round(1)

## 5. Какие факторы менее важны?

Корреляции не доказывают причинность, но помогают увидеть, какие признаки стоит взять в следующую модель.

In [ ]:
numeric_cols = ['price', 'accommodates', 'bedrooms', 'bathrooms', 'beds',
                'minimum_nights', 'number_of_reviews', 'review_scores_rating',
                'availability_365']
corr = df_clean[numeric_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.scatterplot(data=df_clean, x='minimum_nights', y='price', alpha=.2, s=16, ax=axes[0], color='#B7554A')
axes[0].set(xlim=(0, 60), title='Долгий минимум проживания связан с меньшей ценой за ночь',
            xlabel='Минимальное число ночей', ylabel='Цена, €')

sns.heatmap(corr, cmap='vlag', center=0, vmin=-1, vmax=1, square=True, ax=axes[1])
axes[1].set_title('Корреляционная матрица')
fig.tight_layout()
fig.savefig(FIG_DIR / '03_min_nights_and_correlations.png', dpi=180, bbox_inches='tight')
plt.show()

corr['price'].sort_values(ascending=False).round(2)

## 6. Выводы и следующий шаг

- **Ключевые факторы:** тип жилья, район, вместимость / число спален и минимальный срок проживания.
- **Слабые прямые сигналы:** рейтинг, количество отзывов и доступность.
- **Следующий шаг:** разделить данные на train/test, построить регуляризованную регрессию или gradient boosting; добавить описание жилья, координаты и временные факторы.